In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import KFold

PROJECT_ROOT = Path.cwd().parent
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

modeling_data = pd.read_csv(
    DATA_INTERIM / "modeling_data.csv",
    dtype={"FIPS": "string"}
)

print("Dataset shape:", modeling_data.shape)
print("Unique FIPS:", modeling_data["FIPS"].nunique())

Dataset shape: (3135, 22)
Unique FIPS: 3135


In [3]:
selected_predictors = [
    "PCT_LACCESS_POP19",
    "PCT_LACCESS_LOWI19",
    "GROCPTH20",
    "CONVSPTH20",
    "FFRPTH20",
    "FSRPTH20",
    "MEDHHINC21",
    "POVRATE21",
    "CHILDPOVRATE21",
    "DEEPPOVRATE21",
    "PC_SNAPBEN22",
    "PCT_65OLDER20",
    "PCT_18YOUNGER20",
    "PCT_NHWHITE20",
    "PCT_NHBLACK20",
    "PCT_HISP20",
    "PCT_NHASIAN20",
    "RECFACPTH20"
]

X = modeling_data[selected_predictors].copy()
y = modeling_data["OBESITY_AdjPrev"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3135, 18)
y shape: (3135,)


In [4]:
RANDOM_STATE = 42

outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [5]:
fold_summary = []

for fold_number, (train_idx, val_idx) in enumerate(
    outer_cv.split(X),
    start=1
):
    fold_summary.append({
        "outer_fold": fold_number,
        "training_count": len(train_idx),
        "validation_count": len(val_idx)
    })

fold_summary = pd.DataFrame(fold_summary)

display(fold_summary)

print(
    "Total validation observations:",
    fold_summary["validation_count"].sum()
)

,outer_fold,training_count,validation_count
0,1,2508,627
1,2,2508,627
2,3,2508,627
3,4,2508,627
4,5,2508,627


Total validation observations: 3135


In [6]:
modeling_data["outer_fold"] = 0

for fold_number, (_, val_idx) in enumerate(
    outer_cv.split(X),
    start=1
):
    modeling_data.loc[val_idx, "outer_fold"] = fold_number

print(
    modeling_data["outer_fold"]
    .value_counts()
    .sort_index()
)

print(
    "\nCounties without fold assignment:",
    (modeling_data["outer_fold"] == 0).sum()
)

print(
    "Unique FIPS:",
    modeling_data["FIPS"].nunique()
)

outer_fold
1    627
2    627
3    627
4    627
5    627
Name: count, dtype: int64

Counties without fold assignment: 0
Unique FIPS: 3135


In [7]:
fold_1_train_idx, fold_1_val_idx = next(outer_cv.split(X))

X_train_outer = X.iloc[fold_1_train_idx].copy()
X_val_outer = X.iloc[fold_1_val_idx].copy()

y_train_outer = y.iloc[fold_1_train_idx].copy()
y_val_outer = y.iloc[fold_1_val_idx].copy()

print("X training:", X_train_outer.shape)
print("X validation:", X_val_outer.shape)
print("y training:", y_train_outer.shape)
print("y validation:", y_val_outer.shape)

X training: (2508, 18)
X validation: (627, 18)
y training: (2508,)
y validation: (627,)


In [8]:
missing_summary_fold1 = pd.DataFrame({
    "missing_count": X_train_outer.isna().sum(),
    "missing_pct": X_train_outer.isna().mean() * 100
}).round(2)

display(missing_summary_fold1)

,missing_count,missing_pct
PCT_LACCESS_POP19,1,0.04
PCT_LACCESS_LOWI19,1,0.04
GROCPTH20,739,29.47
CONVSPTH20,234,9.33
FFRPTH20,376,14.99
FSRPTH20,226,9.01
MEDHHINC21,1,0.04
POVRATE21,1,0.04
CHILDPOVRATE21,1,0.04
DEEPPOVRATE21,0,0.00


In [9]:
missing_threshold = 20.0

excluded_missing_fold1 = missing_summary_fold1[
    missing_summary_fold1["missing_pct"] > missing_threshold
].index.tolist()

retained_after_missing_fold1 = [
    col for col in X_train_outer.columns
    if col not in excluded_missing_fold1
]

print("Excluded (>20% missing):")
print(excluded_missing_fold1)

print("\nNumber excluded:", len(excluded_missing_fold1))
print("Number retained:", len(retained_after_missing_fold1))

Excluded (>20% missing):
['GROCPTH20', 'RECFACPTH20']

Number excluded: 2
Number retained: 16


In [10]:
X_train_fold1 = X_train_outer[
    retained_after_missing_fold1
].copy()

X_val_fold1 = X_val_outer[
    retained_after_missing_fold1
].copy()

print("Training shape:", X_train_fold1.shape)
print("Validation shape:", X_val_fold1.shape)

print("\nSame columns:",
      X_train_fold1.columns.tolist() ==
      X_val_fold1.columns.tolist())

Training shape: (2508, 16)
Validation shape: (627, 16)

Same columns: True


In [11]:
corr_matrix_fold1 = X_train_fold1.corr(method="pearson")

high_corr_pairs_fold1 = []

columns = corr_matrix_fold1.columns

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        r = corr_matrix_fold1.iloc[i, j]

        if abs(r) >= 0.80:
            high_corr_pairs_fold1.append({
                "predictor_1": columns[i],
                "predictor_2": columns[j],
                "r": r,
                "abs_r": abs(r)
            })

high_corr_pairs_fold1 = pd.DataFrame(high_corr_pairs_fold1)

if not high_corr_pairs_fold1.empty:
    high_corr_pairs_fold1 = (
        high_corr_pairs_fold1
        .sort_values("abs_r", ascending=False)
        .reset_index(drop=True)
    )

display(high_corr_pairs_fold1)

print(
    "Highly correlated pairs:",
    len(high_corr_pairs_fold1)
)

,predictor_1,predictor_2,r,abs_r
0,POVRATE21,CHILDPOVRATE21,0.936780,0.936780
1,PCT_LACCESS_POP19,PCT_LACCESS_LOWI19,0.882368,0.882368


Highly correlated pairs: 2


In [12]:
for _, row in high_corr_pairs_fold1.iterrows():
    p1 = row["predictor_1"]
    p2 = row["predictor_2"]

    print(f"{p1} vs {p2}")
    print(
        f"  {p1}: "
        f"{X_train_outer[p1].isna().mean() * 100:.2f}% missing"
    )
    print(
        f"  {p2}: "
        f"{X_train_outer[p2].isna().mean() * 100:.2f}% missing"
    )
    print()

POVRATE21 vs CHILDPOVRATE21
  POVRATE21: 0.04% missing
  CHILDPOVRATE21: 0.04% missing

PCT_LACCESS_POP19 vs PCT_LACCESS_LOWI19
  PCT_LACCESS_POP19: 0.04% missing
  PCT_LACCESS_LOWI19: 0.04% missing



In [13]:
for predictor in [
    "POVRATE21",
    "CHILDPOVRATE21",
    "PCT_LACCESS_POP19",
    "PCT_LACCESS_LOWI19"
]:
    correlations = (
        corr_matrix_fold1[predictor]
        .drop(predictor)
        .abs()
        .sort_values(ascending=False)
    )

    print(f"\n{predictor}")
    print(correlations.head(5))


POVRATE21
CHILDPOVRATE21    0.936780
MEDHHINC21        0.770761
DEEPPOVRATE21     0.750186
PC_SNAPBEN22      0.654821
PCT_NHBLACK20     0.486360
Name: POVRATE21, dtype: float64

CHILDPOVRATE21
POVRATE21        0.936780
MEDHHINC21       0.780406
DEEPPOVRATE21    0.675635
PC_SNAPBEN22     0.668470
PCT_NHBLACK20    0.500481
Name: CHILDPOVRATE21, dtype: float64

PCT_LACCESS_POP19
PCT_LACCESS_LOWI19    0.882368
FFRPTH20              0.211468
FSRPTH20              0.141651
PC_SNAPBEN22          0.141284
PCT_HISP20            0.089523
Name: PCT_LACCESS_POP19, dtype: float64

PCT_LACCESS_LOWI19
PCT_LACCESS_POP19    0.882368
MEDHHINC21           0.239569
CHILDPOVRATE21       0.230515
POVRATE21            0.222645
PCT_NHWHITE20        0.181752
Name: PCT_LACCESS_LOWI19, dtype: float64


In [14]:
correlation_excluded_fold1 = [
    "CHILDPOVRATE21",
    "PCT_LACCESS_POP19"
]

retained_after_correlation_fold1 = [
    col for col in X_train_fold1.columns
    if col not in correlation_excluded_fold1
]

X_train_fold1 = X_train_fold1[
    retained_after_correlation_fold1
].copy()

X_val_fold1 = X_val_fold1[
    retained_after_correlation_fold1
].copy()

print("Excluded due to correlation:")
print(correlation_excluded_fold1)

print("\nPredictors remaining:", len(retained_after_correlation_fold1))
print("Training shape:", X_train_fold1.shape)
print("Validation shape:", X_val_fold1.shape)

Excluded due to correlation:
['CHILDPOVRATE21', 'PCT_LACCESS_POP19']

Predictors remaining: 14
Training shape: (2508, 14)
Validation shape: (627, 14)


In [15]:
print("Training missing values:",
      X_train_fold1.isna().sum().sum())

print("Validation missing values:",
      X_val_fold1.isna().sum().sum())

display(
    pd.DataFrame({
        "train_missing": X_train_fold1.isna().sum(),
        "validation_missing": X_val_fold1.isna().sum()
    })
)

Training missing values: 881
Validation missing values: 213


,train_missing,validation_missing
PCT_LACCESS_LOWI19,1,1
CONVSPTH20,234,50
FFRPTH20,376,95
FSRPTH20,226,56
MEDHHINC21,1,0
POVRATE21,1,0
DEEPPOVRATE21,0,0
PC_SNAPBEN22,42,11
PCT_65OLDER20,0,0
PCT_18YOUNGER20,0,0


In [16]:
from sklearn.impute import SimpleImputer

In [17]:
imputer_fold1 = SimpleImputer(strategy="median")

# Learn medians ONLY from outer training data
imputer_fold1.fit(X_train_fold1)

# Apply those same medians to both sets
X_train_imputed_fold1 = pd.DataFrame(
    imputer_fold1.transform(X_train_fold1),
    columns=X_train_fold1.columns,
    index=X_train_fold1.index
)

X_val_imputed_fold1 = pd.DataFrame(
    imputer_fold1.transform(X_val_fold1),
    columns=X_val_fold1.columns,
    index=X_val_fold1.index
)

print(
    "Training missing after imputation:",
    X_train_imputed_fold1.isna().sum().sum()
)

print(
    "Validation missing after imputation:",
    X_val_imputed_fold1.isna().sum().sum()
)

print("Training shape:", X_train_imputed_fold1.shape)
print("Validation shape:", X_val_imputed_fold1.shape)

Training missing after imputation: 0
Validation missing after imputation: 0
Training shape: (2508, 14)
Validation shape: (627, 14)


In [18]:
imputation_values_fold1 = pd.Series(
    imputer_fold1.statistics_,
    index=X_train_fold1.columns,
    name="training_median"
)

display(imputation_values_fold1)

PCT_LACCESS_LOWI19        6.627188
CONVSPTH20                0.519900
FFRPTH20                  0.654830
FSRPTH20                  0.693859
MEDHHINC21            56680.000000
POVRATE21                13.500000
DEEPPOVRATE21             5.680450
PC_SNAPBEN22             26.789252
PCT_65OLDER20            19.802754
PCT_18YOUNGER20          21.980256
PCT_NHWHITE20            80.957533
PCT_NHBLACK20             1.937421
PCT_HISP20                4.660359
PCT_NHASIAN20             0.541156
Name: training_median, dtype: float64

In [19]:
from sklearn.preprocessing import StandardScaler

In [20]:
scaler_fold1 = StandardScaler()

# Learn mean and standard deviation from training data only
scaler_fold1.fit(X_train_imputed_fold1)

# Scale training and validation using the training parameters
X_train_scaled_fold1 = pd.DataFrame(
    scaler_fold1.transform(X_train_imputed_fold1),
    columns=X_train_imputed_fold1.columns,
    index=X_train_imputed_fold1.index
)

X_val_scaled_fold1 = pd.DataFrame(
    scaler_fold1.transform(X_val_imputed_fold1),
    columns=X_val_imputed_fold1.columns,
    index=X_val_imputed_fold1.index
)

print("Scaled training shape:", X_train_scaled_fold1.shape)
print("Scaled validation shape:", X_val_scaled_fold1.shape)

print("\nTraining means after scaling:")
print(X_train_scaled_fold1.mean().round(3))

print("\nTraining standard deviations after scaling:")
print(X_train_scaled_fold1.std(ddof=0).round(3))

Scaled training shape: (2508, 14)
Scaled validation shape: (627, 14)

Training means after scaling:
PCT_LACCESS_LOWI19   -0.0
CONVSPTH20            0.0
FFRPTH20             -0.0
FSRPTH20              0.0
MEDHHINC21           -0.0
POVRATE21             0.0
DEEPPOVRATE21         0.0
PC_SNAPBEN22          0.0
PCT_65OLDER20        -0.0
PCT_18YOUNGER20      -0.0
PCT_NHWHITE20         0.0
PCT_NHBLACK20         0.0
PCT_HISP20           -0.0
PCT_NHASIAN20        -0.0
dtype: float64

Training standard deviations after scaling:
PCT_LACCESS_LOWI19    1.0
CONVSPTH20            1.0
FFRPTH20              1.0
FSRPTH20              1.0
MEDHHINC21            1.0
POVRATE21             1.0
DEEPPOVRATE21         1.0
PC_SNAPBEN22          1.0
PCT_65OLDER20         1.0
PCT_18YOUNGER20       1.0
PCT_NHWHITE20         1.0
PCT_NHBLACK20         1.0
PCT_HISP20            1.0
PCT_NHASIAN20         1.0
dtype: float64


In [21]:
iqr_summary_fold1 = []

for column in X_train_fold1.columns:
    values = X_train_fold1[column].dropna()

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (
        (values < lower_bound) |
        (values > upper_bound)
    )

    iqr_summary_fold1.append({
        "predictor": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_mask.sum(),
        "min": values.min(),
        "max": values.max()
    })

iqr_summary_fold1 = pd.DataFrame(iqr_summary_fold1)

display(
    iqr_summary_fold1.round(3)
)

,predictor,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,min,max
0,PCT_LACCESS_LOWI19,3.633,10.835,7.201,-7.169,21.637,131,0.000,62.600
1,CONVSPTH20,0.396,0.689,0.293,-0.043,1.129,70,0.129,4.442
2,FFRPTH20,0.510,0.797,0.288,0.078,1.229,55,0.146,6.684
3,FSRPTH20,0.528,0.899,0.371,-0.028,1.455,155,0.131,9.358
4,MEDHHINC21,49200.500,65648.000,16447.500,24529.250,90319.250,110,25653.000,153716.000
5,POVRATE21,10.600,17.400,6.800,0.400,27.600,67,2.900,43.900
6,DEEPPOVRATE21,4.177,7.513,3.336,-0.826,12.517,108,0.000,36.884
7,PC_SNAPBEN22,16.948,40.630,23.682,-18.576,76.154,52,1.154,259.966
8,PCT_65OLDER20,17.282,22.662,5.380,9.212,30.732,72,4.799,58.860
9,PCT_18YOUNGER20,20.039,23.807,3.767,14.389,29.457,80,6.979,40.882


In [22]:
print(
    "Total values flagged by IQR:",
    iqr_summary_fold1["outlier_count"].sum()
)

print(
    "Predictors with at least one IQR outlier:",
    (iqr_summary_fold1["outlier_count"] > 0).sum()
)

Total values flagged by IQR: 1862
Predictors with at least one IQR outlier: 14


In [23]:
iqr_flagged_values_fold1 = int(
    iqr_summary_fold1["outlier_count"].sum()
)

iqr_predictors_flagged_fold1 = int(
    (iqr_summary_fold1["outlier_count"] > 0).sum()
)

implausible_values_removed_fold1 = 0

print("IQR-flagged values:", iqr_flagged_values_fold1)
print("Predictors with IQR flags:", iqr_predictors_flagged_fold1)
print("Implausible values removed:", implausible_values_removed_fold1)
print("Training counties retained:", len(X_train_fold1))

IQR-flagged values: 1862
Predictors with IQR flags: 14
Implausible values removed: 0
Training counties retained: 2508


### Fold 1 Outlier Decision

The 1.5 × IQR rule flagged 1,862 values across all 14 retained predictors. 
The flagged values were reviewed as potential outliers rather than automatically 
removed. No values were identified as clearly implausible based on their 
variable definitions and observed ranges. Therefore, no observations were 
removed during the Fold 1 outlier screening.